# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset is based on a Croissant schema and contains multiple record sets, fields, and column-level details from clinical, hormonal, imaging, and genetic data.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name: {}\nDescription: {}\n".format(metadata.get('name', 'N/A'), metadata.get('description', 'N/A')))

# Print publication date and keywords
print("Published: {}".format(metadata.get('datePublished', 'N/A')))
print("Keywords:", metadata.get('keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` fields to ensure consistency.

In [ ]:
# List all available record sets and their @id
record_sets = dataset.metadata.record_sets

print("Available record sets:")
for rs in record_sets:
    print("- @id:", rs.id)
    print("  Name:", rs.name)
    print("  Description:", rs.description)
    print("  Fields:")
    for field in rs.fields:
        print("    - @id:", field.id, "| Name:", field.name, "| Data type:", field.data_type)
    print()

# Show some sample records from each record set
for rs in record_sets:
    print(f"Sample records from record set @id: {rs.id}")
    for x in dataset.records(record_set=rs.id):
        print(x)
        break  # Show only the first record as sample
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather all record set @ids
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print column names of clinical record set
print("Columns for each DataFrame:")
for record_set_id, df in dataframes.items():
    print(f"Record set: {record_set_id}")
    print(df.columns.tolist())
    print(df.head(), "\n")

# Choose first record set for analysis
primary_rs_id = record_sets_ids[0] if record_sets_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming distributions, and grouping by attributes.

### Example workflow:
1. Choose a numeric field from a record set (`@id`).
2. Filter records based on a threshold.
3. Normalize the numeric field.
4. Group by a categorical field.

In [ ]:
# Select a numeric field for analysis from the first record set
if primary_rs_id:
    df = dataframes[primary_rs_id]
    # Find a numeric field
    numeric_fields = []
    for field in dataset.metadata.record_sets[0].fields:
        if field.data_type in ['Float', 'Integer', 'Number']:
            numeric_fields.append(field.id)

    print("Numeric fields found:", numeric_fields)

    # Use first numeric field for demonstration
    numeric_field = numeric_fields[0] if numeric_fields else None
    threshold = 10

    if numeric_field and numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

        # Find a group field (categorical)
        group_fields = []
        for field in dataset.metadata.record_sets[0].fields:
            if field.data_type in ['Text', 'Boolean']:
                group_fields.append(field.id)
        group_field = group_fields[0] if group_fields else None

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field present in this record set or not found in DataFrame.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

Example: Plot histogram of numeric fields or scatter plots between clinical features and hormone levels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For demonstration, plot histogram for numeric field in primary record set
if primary_rs_id and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field], bins=10, kde=True)
    plt.title(f"Histogram of {numeric_field} in {primary_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# If multiple numeric fields exist, visualize correlations
if primary_rs_id and len(numeric_fields) > 1:
    plt.figure(figsize=(6,6))
    sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
    plt.title(f"Scatter Plot: {numeric_fields[0]} vs {numeric_fields[1]}")
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, and analysis of the FAIR^2 clinical dataset using `mlcroissant`. Records and fields are referenced by their `@id` values, ensuring traceability and reproducibility across clinical, hormonal, imaging, and genetic domains.

- The dataset is small (N=3), with rich tabular structure and well-documented Croissant schema.
- Data processing steps are possible for exploring genotype-phenotype correlations, filtering by clinical attributes, and visualizing hormone distributions.
- For full-scale ML/AI usage, larger and more diverse datasets are recommended.

For more complex processing, refer to official `mlcroissant` documentation and the dataset's FAIR^2 schema for additional entity definitions and metadata.